# Exp-A/B/C/DXY/Spot Feature Comparison Experiment (Updated)

This notebook compares several feature engineering ideas against the baseline.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import warnings
warnings.filterwarnings("ignore")
sys.path.append("..")

from src.processing import load_and_clean_data
from src.features import generate_features, frac_diff
from src.pooling import pool_boj_data
from src.modeling import walk_forward_validation, summarize_ic

# Load data
df_raw = load_and_clean_data("../data/BOJ_data.xlsx", "../data/BOJ_meeting_history.csv")
print(f"Data loaded. Shape: {df_raw.shape}")
print(f"Columns in df_raw: {df_raw.columns.tolist()}")

Data loaded. Shape: (4653, 30)
Columns in df_raw: ['Date', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'T12', 'T18', 'T24', 'USDJPY', 'JGB_Future', 'Nikkei225', 'DXY', 'Actual_Policy_Rate', 'Is_Meeting_Day', 'Days_to_MPM', 'M1_is_imputed', 'M2_is_imputed', 'M3_is_imputed', 'M4_is_imputed', 'M5_is_imputed', 'M6_is_imputed', 'M7_is_imputed', 'M8_is_imputed', 'T12_is_imputed', 'T18_is_imputed', 'T24_is_imputed']


## Baseline (42 features)

In [2]:
# 1. Baseline
df_feat_base = generate_features(df_raw)  # Default: all flags False
df_pooled_base = pool_boj_data(df_feat_base)

res_3d_base = walk_forward_validation(df_pooled_base, "Target_3d_norm", "2024-01-01")
res_5d_base = walk_forward_validation(df_pooled_base, "Target_5d_norm", "2024-01-01")

ic_3d_base = summarize_ic(res_3d_base)
ic_5d_base = summarize_ic(res_5d_base)

print("Baseline IC (3d):", round(ic_3d_base["ic_all"], 4))
print("Baseline IC (5d):", round(ic_5d_base["ic_all"], 4))

Baseline IC (3d): 0.2252
Baseline IC (5d): 0.2364


## Exp-A: Weekday Cyclic Encoding

In [3]:
# Exp-A
df_feat_a = generate_features(df_raw, add_weekday_cyclic=True)
df_pooled_a = pool_boj_data(df_feat_a)

res_3d_a = walk_forward_validation(df_pooled_a, "Target_3d_norm", "2024-01-01")
res_5d_a = walk_forward_validation(df_pooled_a, "Target_5d_norm", "2024-01-01")

ic_3d_a = summarize_ic(res_3d_a)
ic_5d_a = summarize_ic(res_5d_a)

print("Exp-A IC (3d):", round(ic_3d_a["ic_all"], 4))
print("Exp-A IC (5d):", round(ic_5d_a["ic_all"], 4))
print("\nExp-A 3d ic_by_fold:")
print(ic_3d_a["ic_by_fold"])
print("\nExp-A 5d ic_by_fold:")
print(ic_5d_a["ic_by_fold"])

Exp-A IC (3d): 0.2504
Exp-A IC (5d): 0.2221

Exp-A 3d ic_by_fold:
{np.int64(0): np.float64(0.2004), np.int64(1): np.float64(0.0173), np.int64(2): np.float64(0.1471), np.int64(3): np.float64(0.2025), np.int64(4): np.float64(0.3041), np.int64(5): np.float64(0.2125), np.int64(6): np.float64(0.3846), np.int64(7): np.float64(0.4334), np.int64(8): np.float64(0.6151)}

Exp-A 5d ic_by_fold:
{np.int64(0): np.float64(0.411), np.int64(1): np.float64(-0.007), np.int64(2): np.float64(-0.0713), np.int64(3): np.float64(0.2951), np.int64(4): np.float64(0.3237), np.int64(5): np.float64(-0.05), np.int64(6): np.float64(0.39), np.int64(7): np.float64(0.5135), np.int64(8): np.float64(0.6356)}


## Exp-DXY: Add DXY_frac_diff

In [4]:
def generate_features_with_dxy(df, d=0.4, window=50):
    feat_df = generate_features(df, d=d, window=window)
    if "DXY" in feat_df.columns:
        feat_df["DXY_frac_diff"] = frac_diff(feat_df["DXY"], d=d, window=window)
        print(f"DXY found. DXY_frac_diff head:\n{feat_df["DXY_frac_diff"].dropna().head()}")
    else:
        print("DXY NOT found in feat_df columns!")
    return feat_df

df_feat_dxy = generate_features_with_dxy(df_raw)
df_pooled_dxy = pool_boj_data(df_feat_dxy)

if "DXY_frac_diff" not in df_pooled_dxy.columns:
    print("DXY_frac_diff missing from pooled, merging manually...")
    dxy_map = df_feat_dxy[["Date", "DXY_frac_diff"]].drop_duplicates()
    df_pooled_dxy = df_pooled_dxy.merge(dxy_map, on="Date", how="left")

print(f"Is DXY_frac_diff in df_pooled_dxy? {"DXY_frac_diff" in df_pooled_dxy.columns}")
if "DXY_frac_diff" in df_pooled_dxy.columns:
    print(f"Non-NaN DXY_frac_diff count in pooled: {df_pooled_dxy["DXY_frac_diff"].notnull().sum()}")

res_3d_dxy = walk_forward_validation(df_pooled_dxy, "Target_3d_norm", "2024-01-01")
res_5d_dxy = walk_forward_validation(df_pooled_dxy, "Target_5d_norm", "2024-01-01")

ic_3d_dxy = summarize_ic(res_3d_dxy)
ic_5d_dxy = summarize_ic(res_5d_dxy)
print("Exp-DXY IC (3d):", round(ic_3d_dxy["ic_all"], 4))
print("Exp-DXY IC (5d):", round(ic_5d_dxy["ic_all"], 4))

DXY found. DXY_frac_diff head:
49    11.814784
50    11.604440
51    11.789718
52    11.743297
53    11.636594
Name: DXY_frac_diff, dtype: float64
Is DXY_frac_diff in df_pooled_dxy? True
Non-NaN DXY_frac_diff count in pooled: 50422


Exp-DXY IC (3d): 0.2252
Exp-DXY IC (5d): 0.2364


## Summary of Results

In [5]:
results = {
    "Baseline": {
        "3d_ic_all": ic_3d_base["ic_all"],
        "3d_ic_recent": ic_3d_base["ic_recent"],
        "5d_ic_all": ic_5d_base["ic_all"],
        "5d_ic_recent": ic_5d_base["ic_recent"],
    },
    "Exp-A (+weekday)": {
        "3d_ic_all": ic_3d_a["ic_all"],
        "3d_ic_recent": ic_3d_a["ic_recent"],
        "5d_ic_all": ic_5d_a["ic_all"],
        "5d_ic_recent": ic_5d_a["ic_recent"],
    },
    "Exp-DXY (+DXY_fd)": {
        "3d_ic_all": ic_3d_dxy["ic_all"],
        "3d_ic_recent": ic_3d_dxy["ic_recent"],
        "5d_ic_all": ic_5d_dxy["ic_all"],
        "5d_ic_recent": ic_5d_dxy["ic_recent"],
    },
}

df_results = pd.DataFrame(results).T
df_results["3d_delta"] = df_results["3d_ic_all"] - df_results.loc["Baseline", "3d_ic_all"]
df_results["5d_delta"] = df_results["5d_ic_all"] - df_results.loc["Baseline", "5d_ic_all"]
print(df_results.round(4).to_string())

                   3d_ic_all  3d_ic_recent  5d_ic_all  5d_ic_recent  3d_delta  5d_delta
Baseline              0.2252        0.4209     0.2364        0.4347    0.0000    0.0000
Exp-A (+weekday)      0.2504        0.4461     0.2221        0.4183    0.0252   -0.0143
Exp-DXY (+DXY_fd)     0.2252        0.4209     0.2364        0.4347    0.0000    0.0000
